# تحليل بيانات Fridge-tag 2E
استخراج البيانات من ملفات PDF الخاصة بأجهزة Fridge-tag 2E وتحويلها إلى جداول قابلة للتحليل.

**ملاحظة:** قبل رفع هذا الملف إلى المستودع (commit)، يجب مسح جميع المخرجات عبر `Kernel → Restart & Clear Output`.

In [ ]:
import pandas as pd
import numpy as np
import re
import pdfplumber
from pathlib import Path
import warnings

# إخفاء التحذيرات غير الضرورية
warnings.filterwarnings('ignore')

# إعداد مسارات المجلدات (قم بتعديلها حسب مسارات مشروعك)
INPUT_FT2_DIR = Path("data/raw_pdfs")
OUTPUT_DIR = Path("data/processed")

# إنشاء المجلدات إذا لم تكن موجودة
INPUT_FT2_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("✅ تم استدعاء المكتبات وإعداد المسارات بنجاح.")

In [ ]:
# 2. دالة تحويل المدة إلى دقائق (معالجة جميع الصيغ: أيام، ساعات، دقائق) - محصنة أمنياً
import re
import pandas as pd
import numpy as np

def safe_duration_to_minutes(dur_str):
    """
    دالة محصنة لتحويل نصوص المدة الزمنية إلى دقائق.
    تم تصميمها لمنع ثغرات (ReDoS) و (Integer Overflow).
    """
    # 1. الدفاع الأول: تقييد طول السلسلة النصية لمنع استنفاد المعالج
    if pd.isna(dur_str) or not isinstance(dur_str, str) or len(dur_str) > 50:
        return np.nan 
        
    total = 0
    
    # 2. الدفاع الثاني: Regex محدد الخانات (بحد أقصى 4 أرقام) لمنع Overflow
    days = re.search(r'(\d{1,4})d', dur_str)
    hours = re.search(r'(\d{1,4})h', dur_str)
    mins = re.search(r'(\d{1,4})min', dur_str)
    
    if days: total += int(days.group(1)) * 1440
    if hours: total += int(hours.group(1)) * 60
    if mins: total += int(mins.group(1))
    
    # 3. الدفاع الثالث: Sanity Check (تجاهل القيم الفلكية الناتجة عن أعطال الحساسات)
    MAX_LOGICAL_MINUTES = 3_000_000 # ما يعادل تقريباً 5 سنوات
    if total > MAX_LOGICAL_MINUTES:
        return np.nan 
        
    return total

print("✅ تم تعريف دالة safe_duration_to_minutes المحصنة بنجاح.")

In [ ]:
# 3. مثال: معالجة ملف PDF واحد (للتوضيح)
all_extracted_data =[]

# محاكاة لاستخراج البيانات من ملفات PDF (استبدل هذا بمنطق pdfplumber الفعلي الخاص بك)
# كمثال أضفت بيانات سليمة وبيانات تالفة لاختبار الحماية
all_extracted_data =[
    {"File": "FT2_001.pdf", "Date": "2023-01-01", "Time_Above": "1d 02h 30min", "Time_Below": "0h 0min"},
    {"File": "FT2_001.pdf", "Date": "2023-01-02", "Time_Above": "0h 45min", "Time_Below": "1h 15min"},
    {"File": "FT2_002.pdf", "Date": "2023-01-03", "Time_Above": "9999999999d", "Time_Below": "0h 0min"} # قيمة تالفة خبيثة
]

# ---------------------------------------------------------
# إنشاء DataFrame (هنا يتم تعريف df_parsed لحل الخطأ السابق)
# ---------------------------------------------------------
df_parsed = pd.DataFrame(all_extracted_data)

print(f"✅ تم تحميل {len(df_parsed)} سجل بنجاح في df_parsed.")
display(df_parsed.head())

In [ ]:
# 4. استخراج الجداول من ملف PDF واحد (تجريبي)
# 1. تحويل المدد إلى دقائق باستخدام الدالة المحصنة (تم توحيد الطريقة للعمودين)
df_parsed["Time_Above_min"] = df_parsed["Time_Above"].apply(safe_duration_to_minutes)

# 2. التحويل الآمن في Pandas (Defensive Casting) للعمود الأول
df_parsed["Time_Above_min"] = pd.to_numeric(
    df_parsed["Time_Above_min"], 
    errors='coerce',  # تحويل القيم الخاطئة أو الفلكية إلى NaN بدلاً من انهيار النظام
    downcast='integer'  # تقليل حجم الذاكرة المستخدمة بأمان
)

# 3. معالجة العمود الثاني (Time_Below)
df_parsed["Time_Below_min"] = df_parsed["Time_Below"].apply(safe_duration_to_minutes)

# 4. التحويل الآمن للعمود الثاني أيضاً
df_parsed["Time_Below_min"] = pd.to_numeric(
    df_parsed["Time_Below_min"], 
    errors='coerce',
    downcast='integer'
)

# 5. التحقق من السجلات التالفة والتنبيه
corrupted_above = df_parsed[df_parsed["Time_Above_min"].isna()]
corrupted_below = df_parsed[df_parsed["Time_Below_min"].isna()]
total_corrupted = len(corrupted_above) + len(corrupted_below)

if total_corrupted > 0:
    print(f"⚠️ تحذير أمني: تم اكتشاف وعزل {total_corrupted} سجلات تالفة أو تحتوي على قيم فلكية غير منطقية.")

print("\n✅ تم إنهاء المعالجة بأمان. النتائج:")
display(df_parsed)

In [ ]:
# 5. معالجة جميع ملفات PDF في مجلد input_ft2 (إكمال الحلقة غير المكتملة سابقاً)
# حفظ البيانات المنظفة في ملف Excel
output_file = OUTPUT_DIR / "FT2_Cleaned_Data.xlsx"
df_parsed.to_excel(output_file, index=False)

print(f"✅ تمت العملية بالكامل! تم حفظ الملف النهائي في: {output_file}")

In [ ]:
            # تحويل المدد إلى دقائق باستخدام الدالة المحصنة (تم توحيد الطريقة للعمودين)
            df_parsed["Time_Above_min"] = df_parsed["Time_Above"].apply(safe_duration_to_minutes)
            
            # التحويل الآمن في Pandas (Defensive Casting)
            df_parsed["Time_Above_min"] = pd.to_numeric(
                df_parsed["Time_Above_min"], 
                errors='coerce',  # تحويل القيم الخاطئة أو الفلكية إلى NaN بدلاً من انهيار النظام
                downcast='integer'  # تقليل حجم الذاكرة المستخدمة بأمان
            )
            
            # Time_Below_min: محفوظ للاستخدام المستقبلي
            # السبب:
            # - دالة safe_duration_to_minutes مصممة أصلاً لمعالجة كلا الاتجاهين (أعلى وأدنى)
            # - في أنظمة مراقبة سلسلة التبريد الدوائية (Fridge-tag 2E) يُطلب عادة تحليل الانحرافات المنخفضة (< 2°C أو الحد الأدنى المسموح)
            # - يُمكن استخدامه لاحقاً في فلترة الإنذارات المنخفضة، تقارير الامتثال الثنائي، أو رسوم بيانية مقارنة
            # - الاحتفاظ به الآن يتجنب إعادة الحساب مستقبلاً ويحافظ على اكتمال معالجة البيانات الخام
            df_parsed["Time_Below_min"] = df_parsed["Time_Below"].apply(safe_duration_to_minutes)
            
            # التحويل الآمن للعمود الثاني أيضاً
            df_parsed["Time_Below_min"] = pd.to_numeric(
                df_parsed["Time_Below_min"], 
                errors='coerce',
                downcast='integer'
            )
            
            # التحقق من السجلات التالفة
            corrupted_above = df_parsed[df_parsed["Time_Above_min"].isna()]
            corrupted_below = df_parsed[df_parsed["Time_Below_min"].isna()]
            total_corrupted = len(corrupted_above) + len(corrupted_below)
            if total_corrupted > 0:
                print(f"⚠️ تحذير: تم اكتشاف {total_corrupted} سجلات تالفة أو غير منطقية في ملف PDF.")
            
            display(df_parsed.head())

In [ ]:
# 7. رسم بياني: تطور أقصى درجة حرارة يومية (إذا توفرت البيانات)
if 'df_parsed' in locals() and not df_parsed.empty:
    df_parsed["Date_dt"] = pd.to_datetime(df_parsed["Date"], format="%d.%m.%Y")
    
    plt.figure(figsize=(12, 5))
    plt.plot(df_parsed["Date_dt"], df_parsed["Max Temp"], marker='o', linestyle='-', linewidth=1)
    plt.axhline(y=8.0, color='r', linestyle='--', label='الحد الأعلى (+8°C)')
    plt.xlabel("التاريخ")
    plt.ylabel("أقصى درجة حرارة (°C)")
    plt.title("تطور أقصى درجة حرارة يومية - جهاز Fridge-tag")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("لا توجد بيانات كافية للرسم. تأكد من معالجة الملف أولاً.")

In [ ]:
# 8. تصفية الأيام التي استوفت شرط الإنذار (>= 600 دقيقة فوق 8°C)
if 'df_parsed' in locals() and not df_parsed.empty:
    alarm_days = df_parsed[df_parsed["Time_Above_min"] >= 600]
    if not alarm_days.empty:
        print("أيام الإنذار (انحراف حراري طويل):")
        display(alarm_days[["Date", "Avg Temp", "Max Temp", "Time_Above_min"]])
    else:
        print("لا توجد أيام بها انحراف حراري طويل (>= 600 دقيقة).")
    
    short_excursions = df_parsed[(df_parsed["Max Temp"] > 8.0) & (df_parsed["Time_Above_min"] < 600)]
    if not short_excursions.empty:
        print("\nانحرافات قصيرة (< 600 دقيقة):")
        display(short_excursions[["Date", "Max Temp", "Time_Above_min"]])
else:
    print("لا توجد بيانات. قم بتشغيل الخلايا السابقة أولاً.")